### Data Ingestion To Vector DB Pipeline

In [63]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
# from langchain.textsplitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [17]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf")

Found 2 PDF files to process

Processing: Lecture 10.pdf
  ✓ Loaded 75 pages

Processing: Lecture 12.pdf
  ✓ Loaded 55 pages

Total documents loaded: 130


In [30]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [31]:
chunked_documents = split_documents(all_pdf_documents)  

Split 130 documents into 163 chunks

Example chunk:
Content: 1
Back-End Development
Web Servers,
Back-end Abstractions,
Building APIs
1...
Metadata: {'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2024-12-04T00:34:22+05:00', 'title': '', 'author': 'DELL', 'moddate': '2024-12-04T00:34:22+05:00', 'source': '..\\data\\pdf\\Lecture 10.pdf', 'total_pages': 75, 'page': 0, 'page_label': '1', 'source_file': 'Lecture 10.pdf', 'file_type': 'pdf', 'chunk_index': 0}


In [32]:
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
import uuid
from chromadb.config import Settings
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


d:\Projects\traditionalRag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [54]:
class EmbeddingManager:
    """Handles Document Embedding Using Sentence Transformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize Embedding Manager
        
        Args: 
            Hugging face model name for sentence transformer (default: "all-MiniLM-L6-v2")
        """
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Dimensions: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def generateEmbeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        
        Args:
            texts: List of strings to embed
        Returns:
            Numpy array of embeddings
        """
        if not self.model:
            raise ValueError("Model not loaded")
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return np.array(embeddings)
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise


In [55]:
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2711.06it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Dimensions: 384


### VectorDatabase


In [ ]:
    def connect_to_db(self):
        """Create persistent chroma db client and collection"""
        try:
            os.makedirs(self.persistent_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persistent_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name)
            print(f"Vector store connected: {self.collection_name}")
            print(f"Collection contains {self.collection.count()} documents")
        except Exception as e:
            print(f"Error connecting to ChromaDB: {e}")
            raise

In [61]:
store = VectorStore()

Error connecting to ChromaDB: Client() got an unexpected keyword argument 'path'


TypeError: Client() got an unexpected keyword argument 'path'